# Notebook 3: Closed-Set and OOD Confidence Evaluation

This notebook evaluates the certainty of the trained baseline models on both in-distribution test data and out-of-distribution (OOD) inputs. The in-distribution test set contains examples from the known classes, while the OOD datasets contain examples outside the classes used to train the models. This comparison shows whether the models assign lower confidence to unfamiliar inputs or remain overconfident when making predictions.

The workflow loads the saved model checkpoints, prepares the evaluation images using the same preprocessing pipeline as training, and runs inference on the in-distribution and OOD datasets. For every image, the model produces a prediction among the known classes and a confidence score based on its output probabilities. The notebook compares these confidence scores, including maximum softmax probabilities, across the in-distribution and OOD groups.

No abstention or rejection rule is applied in this notebook. OOD examples are therefore forced into one of the known classes, allowing the experiment to measure the models' raw certainty and expose overconfident OOD predictions. The resulting logits, predictions, and confidence summaries provide the baseline for the later notebooks, where abstention methods can be introduced and evaluated.

## Import Libraries

In [25]:
from pathlib import Path
import json
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    mobilenet_v3_small,
    efficientnet_b0,
)
from PIL import Image
import numpy as np

In [26]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", device)

Device: mps


## Load Test ID dataset and OOD dataset Manifests

In [27]:
SPLIT_DIR = Path("./splits")

test_id = pd.read_csv(SPLIT_DIR / "test_id.csv")
ood_grape = pd.read_csv(SPLIT_DIR / "ood_grape.csv")
ood_corn = pd.read_csv(SPLIT_DIR / "ood_corn.csv")
ood_nonplant = pd.read_csv(SPLIT_DIR / "ood_nonplant.csv")

with open(SPLIT_DIR / "class_to_idx.json", "r") as f:
    class_to_idx = json.load(f)

idx_to_class = {
    idx: class_name
    for class_name, idx in class_to_idx.items()
}

NUM_CLASSES = len(class_to_idx)

print("Classes:", NUM_CLASSES)
print("ID test:", len(test_id))
print("Grape OOD:", len(ood_grape))
print("Corn OOD:", len(ood_corn))
print("Non-plant OOD:", len(ood_nonplant))

Classes: 30
ID test: 9159
Grape OOD: 4062
Corn OOD: 3852
Non-plant OOD: 600


## Create Dataset

In [28]:
class InferenceDataset(Dataset):
    def __init__(self, dataframe, transform=None, has_label=True):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.has_label = has_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        if self.has_label:
            label = class_to_idx[row["class_name"]]
            return image, label, row["path"]

        return image, row["path"]

In [29]:
# Transformation for pre-trained model's expected input format
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [30]:
# Create dataset objects for test, grape OOD and corn OOD

test_dataset = InferenceDataset(
    test_id,
    transform=eval_transform,
    has_label=True,
)

grape_dataset = InferenceDataset(
    ood_grape,
    transform=eval_transform,
    has_label=False,
)

corn_dataset = InferenceDataset(
    ood_corn,
    transform=eval_transform,
    has_label=False,
)

nonplant_dataset = InferenceDataset(
    ood_nonplant,
    transform=eval_transform,
    has_label=False,
)

In [31]:
# Create data loaders objects for test, grape OOD and corn OOD

BATCH_SIZE = 64

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

grape_loader = DataLoader(
    grape_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

corn_loader = DataLoader(
    corn_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

nonplant_loader = DataLoader(
    nonplant_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

## Recreate Model Architectures

In [32]:
def create_model(model_name, num_classes):
    if model_name == "mobilenet_v3_small":
        model = mobilenet_v3_small(weights=None)
        model.classifier[3] = nn.Linear(
            model.classifier[3].in_features,
            num_classes,
        )

    elif model_name == "efficientnet_b0":
        model = efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            num_classes,
        )

    else:
        raise ValueError(model_name)

    return model

In [33]:
MODEL_DIR = Path("./models")

# Load mobilenet weights
mobilenet = create_model(
    "mobilenet_v3_small",
    NUM_CLASSES,
)

mobilenet.load_state_dict(
    torch.load(
        MODEL_DIR / "mobilenet_v3_small_seed42.pt",
        map_location="cpu",
    )
)

mobilenet = mobilenet.to(device)
mobilenet.eval()

# Load efficientnet weights
efficientnet = create_model(
    "efficientnet_b0",
    NUM_CLASSES,
)

efficientnet.load_state_dict(
    torch.load(
        MODEL_DIR / "efficientnet_b0_seed42.pt",
        map_location="cpu",
    )
)

efficientnet = efficientnet.to(device)
efficientnet.eval()

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          

## Generic Inference Function

In [34]:
@torch.no_grad()
def infer_id(model, loader, device):
    all_logits = []
    all_labels = []
    all_paths = []

    model.eval()

    for images, labels, paths in loader:
        images = images.to(device)

        logits = model(images)

        all_logits.append(logits.cpu())
        all_labels.extend(labels.numpy())
        all_paths.extend(paths)

    return {
        "logits": torch.cat(all_logits),
        "labels": torch.tensor(all_labels),
        "paths": all_paths,
    }

@torch.no_grad()
def infer_ood(model, loader, device):
    all_logits = []
    all_paths = []

    model.eval()

    for images, paths in loader:
        images = images.to(device)

        logits = model(images)

        all_logits.append(logits.cpu())
        all_paths.extend(paths)

    return {
        "logits": torch.cat(all_logits),
        "paths": all_paths,
    }

## Run Inference on Test ID and OOD Datasets

In [35]:
# MobileNetV3-Small

mobile_id = infer_id(
    mobilenet,
    test_loader,
    device,
)

mobile_grape = infer_ood(
    mobilenet,
    grape_loader,
    device,
)

mobile_corn = infer_ood(
    mobilenet,
    corn_loader,
    device,
)

mobile_nonplant = infer_ood(
    mobilenet,
    nonplant_loader,
    device,
)

In [36]:
# EfficientNet-B0

efficient_id = infer_id(
    efficientnet,
    test_loader,
    device,
)

efficient_grape = infer_ood(
    efficientnet,
    grape_loader,
    device,
)

efficient_corn = infer_ood(
    efficientnet,
    corn_loader,
    device,
)

efficient_nonplant = infer_ood(
    efficientnet,
    nonplant_loader,
    device,
)

## Compare Confidence of Outputs

### MobileNetV3-Small

In [37]:
# In-distribution Test Data
id_probs = torch.softmax(
    mobile_id["logits"],
    dim=1,
)

id_confidence, id_pred_idx = id_probs.max(dim=1)

mobile_id_results = pd.DataFrame({
    "path": mobile_id["paths"],
    "confidence": id_confidence.numpy(),
    "predicted_class": [
        idx_to_class[int(i)]
        for i in id_pred_idx
    ],
})

# Out-of-distribution Grape Data
ood_grape_probs = torch.softmax(
    mobile_grape["logits"],
    dim=1,
)

ood_grape_confidence, ood_grape_pred_idx = ood_grape_probs.max(dim=1)

mobile_grape_results = pd.DataFrame({
    "path": mobile_grape["paths"],
    "confidence": ood_grape_confidence.numpy(),
    "predicted_class": [
        idx_to_class[int(i)]
        for i in ood_grape_pred_idx
    ],
})

# Out-of-distribution Corn Data
ood_corn_probs = torch.softmax(
    mobile_corn["logits"],
    dim=1,
)

ood_corn_confidence, ood_corn_pred_idx = ood_corn_probs.max(dim=1)

mobile_corn_results = pd.DataFrame({
    "path": mobile_corn["paths"],
    "confidence": ood_corn_confidence.numpy(),
    "predicted_class": [
        idx_to_class[int(i)]
        for i in ood_corn_pred_idx
    ],
})

# Out-of-distribution Non-plant
ood_nonplant_probs = torch.softmax(
    mobile_nonplant["logits"],
    dim=1,
)

ood_nonplant_confidence, ood_nonplant_pred_idx = ood_nonplant_probs.max(dim=1)

mobile_nonplant_results = pd.DataFrame({
    "path": mobile_nonplant["paths"],
    "confidence": ood_nonplant_confidence.numpy(),
    "predicted_class": [
        idx_to_class[int(i)]
        for i in ood_nonplant_pred_idx
    ],
})

In [38]:
closed_set_mobilenet_summary = pd.DataFrame([
    {
        "dataset": "ID",
        "mean_max_softmax": float(id_confidence.mean()),
    },
    {
        "dataset": "Grape OOD",
        "mean_max_softmax": float(ood_grape_confidence.mean()),
    },
    {
        "dataset": "Corn OOD",
        "mean_max_softmax": float(ood_corn_confidence.mean()),
    },
        {
        "dataset": "Non-plant OOD",
        "mean_max_softmax": float(ood_nonplant_confidence.mean()),
    }
])

closed_set_mobilenet_summary

,dataset,mean_max_softmax
0,ID,0.992644
1,Grape OOD,0.782586
2,Corn OOD,0.700715
3,Non-plant OOD,0.709370


### EfficientNet-B0

In [39]:
# In-distribution Test Data
id_probs = torch.softmax(
    efficient_id["logits"],
    dim=1,
)

id_confidence, id_pred_idx = id_probs.max(dim=1)

efficient_id_results = pd.DataFrame({
    "path": efficient_id["paths"],
    "confidence": id_confidence.numpy(),
    "predicted_class": [
        idx_to_class[int(i)]
        for i in id_pred_idx
    ],
})

# Out-of-distribution Grape Data
ood_grape_probs = torch.softmax(
    efficient_grape["logits"],
    dim=1,
)

ood_grape_confidence, ood_grape_pred_idx = ood_grape_probs.max(dim=1)

efficient_grape_results = pd.DataFrame({
    "path": efficient_grape["paths"],
    "confidence": ood_grape_confidence.numpy(),
    "predicted_class": [
        idx_to_class[int(i)]
        for i in ood_grape_pred_idx
    ],
})

# Out-of-distribution Corn Data
ood_corn_probs = torch.softmax(
    efficient_corn["logits"],
    dim=1,
)

ood_corn_confidence, ood_corn_pred_idx = ood_corn_probs.max(dim=1)

efficient_corn_results = pd.DataFrame({
    "path": efficient_corn["paths"],
    "confidence": ood_corn_confidence.numpy(),
    "predicted_class": [
        idx_to_class[int(i)]
        for i in ood_corn_pred_idx
    ],
})

# Out-of-distribution Non-plant
ood_nonplant_probs = torch.softmax(
    efficient_nonplant["logits"],
    dim=1,
)

ood_nonplant_confidence, ood_nonplant_pred_idx = ood_nonplant_probs.max(dim=1)

efficient_nonplant_results = pd.DataFrame({
    "path": efficient_nonplant["paths"],
    "confidence": ood_nonplant_confidence.numpy(),
    "predicted_class": [
        idx_to_class[int(i)]
        for i in ood_nonplant_pred_idx
    ],
})

In [40]:
closed_set_efficientnet_summary = pd.DataFrame([
    {
        "dataset": "ID",
        "mean_max_softmax": float(id_confidence.mean()),
    },
    {
        "dataset": "Grape OOD",
        "mean_max_softmax": float(ood_grape_confidence.mean()),
    },
    {
        "dataset": "Corn OOD",
        "mean_max_softmax": float(ood_corn_confidence.mean()),
    },
    {
        "dataset": "Non-plant OOD",
        "mean_max_softmax": float(ood_nonplant_confidence.mean()),
    }

])

closed_set_efficientnet_summary

,dataset,mean_max_softmax
0,ID,0.996387
1,Grape OOD,0.739006
2,Corn OOD,0.683056
3,Non-plant OOD,0.632370


## Save Logits
Useful to save for easy abstention mechanisms tests

In [44]:
LOGIT_DIR = Path("./logits")
LOGIT_DIR.mkdir(exist_ok=True)

In [45]:
torch.save(
    mobile_id,
    LOGIT_DIR / "mobilenet_id_seed42.pt",
)

torch.save(
    mobile_grape,
    LOGIT_DIR / "mobilenet_grape_seed42.pt",
)

torch.save(
    mobile_corn,
    LOGIT_DIR / "mobilenet_corn_seed42.pt",
)

torch.save(
    mobile_nonplant,
    LOGIT_DIR / "mobilenet_nonplant_seed42.pt",
)

In [46]:
torch.save(
    efficient_id,
    LOGIT_DIR / "efficientnet_id_seed42.pt",
)

torch.save(
    efficient_grape,
    LOGIT_DIR / "efficientnet_grape_seed42.pt",
)

torch.save(
    efficient_corn,
    LOGIT_DIR / "efficientnet_corn_seed42.pt",
)

torch.save(
    efficient_nonplant,
    LOGIT_DIR / "efficientnet_nonplant_seed42.pt",
)